# SENTIMENT ANAYLYSIS

In [1]:
import pandas as pd

## 1. loading data

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape #(50000 , 1)
df.info() #both object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [4]:
# check null 
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
# check duplicate
df.drop_duplicates(inplace=True) #if duplicate exists then drop , and change shape

In [6]:
df.shape

(49582, 2)

## 2. Preprocessing

### a . converting to lowercase 


In [7]:
df["review"] = df["review"].str.lower() 

In [8]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### b. removing urls

In [9]:
import re 

# ##example
# sample_text = "abc is the word, abc"  #convert abc-> xyz
# new_text = re.sub("abc" , "xyz" , sample_text) 
# print(new_text)

In [10]:
def remove_url(text) :
    # ex =  https://docs.python.org/3/library/re.html#functions
    text = re.sub( r"http\S+", "" , text  ) ## (pattern , replacement , string)
    return text 
    
# cal function 
df["review"] = df["review"].apply(remove_url) 

### c . remove punctuations 

In [11]:
def remove_punctuations(text):
    # A-Z a-z 0-9 \s" (space) nhi hai -- punctuations h 
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text)
    return text
    
df["review"] = df["review"].apply(remove_punctuations) 

### d. remove HTML tags

In [12]:
def remove_html(text):
   #  . => any character , * => zero or more char , ? => non greedy 
    text = re.sub(r"<.*?>" , "" , text)
    return text
    
df["review"] = df["review"].apply(remove_html) 

### e. removing stop words

In [13]:
import nltk  

nltk.download("punkt") #official tokenization in nltk 
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize 
from nltk.corpus import stopwords 

In [15]:
# # example
# sample = "I love coding!"
# tokens = word_tokenize(sample)
# tokens

In [16]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english") #to get all stopwords of english lang

    # check if any stopword exists in token , remove them from the text 
    for word in tokens :
         if word in stop_words :
             text = text.replace(word  ,"") 

    return text 

df["review"] = df["review"].apply(remove_stopwords) 


In [17]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### f. Stemming

In [18]:
# running - run
from nltk.stem import PorterStemmer

In [19]:
def stemming (text):
    ps = PorterStemmer() #object
    stemmed_words = [] #list to store words after stemming

    tokens = word_tokenize(text) 

    for token in tokens :
        stemmed_token = ps.stem(token) 
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)  #returned a string of stemmed words


df["review"] = df["review"].apply(stemming) 


In [20]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### g. Encoding of target value


In [21]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [22]:
y = df["sentiment"]
y 

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### h. Vectorization 

In [24]:
df.head() 
# review data is text --- convert into number -- vectors

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [27]:
# TF-IDF  
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"]) 

In [28]:
X  #4million data

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## 3. Dataset and DataLoader

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test , y_train, y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42 
)

In [35]:
X_train.shape

(39665, 5000)

In [39]:
import torch 
from torch.utils.data import TensorDataset , DataLoader 

In [37]:
# X_train = sparse matrix ==> numpy array 
X_train = X_train.toarray()
X_test = X_test.toarray()

In [40]:
train_set = TensorDataset(
   torch.from_numpy(X_train).float() , #convert numpy arry into tensor
   torch.from_numpy(y_train.values).float() 
)
test_set = TensorDataset(
   torch.from_numpy(X_test).float() , #convert numpy arry into tensor
   torch.from_numpy(y_test.values).float() 
)


In [41]:
train_loader = DataLoader(train_set , batch_size = 64 , shuffle=True )
test_loader = DataLoader(test_set , batch_size = 64 , shuffle=True )

## 4. Build an RNN 

In [42]:
import torch.nn as nn 
import torch.optim as optim

In [46]:
# many to one architecture

class RNN(nn.Module):
    def __init__(self , input_size , hidden_size= 128 ,num_layers = 1):
        super(RNN ,self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # rnn layer -- defined in pytorch -- not to build from scratch
        self.rnn = nn.RNN(input_size , hidden_size , num_layers , batch_first=True)
                    # input = 39665
                    # batch_first == shape changing of input  

        # fully connected layer
        self.fc = nn.Linear(hidden_size , 1) #(ip_size , op_size)

    def forward(self , x):
        # optional => hidden state  => by default 0 
        # shape (n_layer , batch_size , hidden size) -- matrix of zeros
        h0 = torch.zeros(self.num_layers, x.size(0) ,  self.hidden_size )

        out, _ = self.rnn(x, h0)
        # return 2 vals =
               # 1st == hidden state of all the timestamp -> (batch , seq_len , hidden_state)
               # 2nd == final hidden state of last timestamp 

        out = self.fc(out[:, -1 ,:]) #(batch , seq_len , hidden_state)  get last timestep
        # out same as prob = 0.2 - negative , 0.8=positive

        return out 

In [50]:
input_size = X_train.shape[1]

model = RNN(input_size)

criteria = nn.BCELoss() #binary cross entropy -- output - 2 class
optimizer = optim.Adam(model.parameters())

## 5. Training RNN

In [54]:
epochs = 10 

for epoch in range(epochs):
    model.train()

    for xb , yb in train_loader :
        optimizer.zero_grad()

        # xb = tf idf = 2d --> model expecting 3d
        xb = xb.unsqueeze(1) #add singleton dimension
        
        output = model.forward(xb)  #(batch_size , 1) = 2d
        #raw outputs -> convert into prob by sigmoid
        output = torch.sigmoid(output.squeeze()) #(batch_size ,) = 1d --> probability
        
        loss = criteria(output , yb) 
        loss.backward()
        optimizer.step() #weight update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()} ")

epoch = 1/10 and loss = 0.17199435830116272 
epoch = 2/10 and loss = 0.2205626219511032 
epoch = 3/10 and loss = 0.3020656108856201 
epoch = 4/10 and loss = 0.11097963154315948 
epoch = 5/10 and loss = 0.1970144510269165 
epoch = 6/10 and loss = 0.2852214574813843 
epoch = 7/10 and loss = 0.19193342328071594 
epoch = 8/10 and loss = 0.3333447575569153 
epoch = 9/10 and loss = 0.11892611533403397 
epoch = 10/10 and loss = 0.1745232194662094 


In [55]:
# evaluate
model.eval()

with torch.no_grad():
    correct_vals = 0 
    total_vals = 0 

    for xb , yb in test_loader :

        xb = xb.unsqueeze(1) 
        output = model.forward(xb) 
        output = torch.sigmoid(output.squeeze()) #convert to probability

        predicted = (output > 0.5).float() #if >0.5 = True.float() = 1.0 and <0.5 = 0 

        total_vals +=  yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"Accuracy = {correct_vals/total_vals *100}")

Accuracy = 85.4189775133609
